# SAC on QuadX-Waypoints-v4

This notebook gathers the final Waypoints SAC workflow in one place. It mirrors the Hover setup with one notebook, one helper, and one grouped reward-variant file.

Included experiment families:
- mode comparison: flight modes `0` and `6` with the report settings
- delayed learning in mode `0`
- reward-shaping trials in mode `0`

Training and report generation are implemented in `training_cell_waypoints_sac.py`, while the different reward formulations are grouped in `waypoints_reward_variants.py`.


## 1. Installs


In [ ]:
# Uncomment if needed:
# %pip install pyflyt stable-baselines3[extra] pandas matplotlib


## 2. Imports and Paths


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "training_cell_waypoints_sac.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    guessed = NOTEBOOK_DIR / "scripts" / "Waypoints"
    if guessed.exists():
        NOTEBOOK_DIR = guessed
        PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
    else:
        PROJECT_ROOT = NOTEBOOK_DIR

SCRIPTS_DIR = PROJECT_ROOT / "scripts"

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Notebook dir:  {NOTEBOOK_DIR}")
print(f"Project root:  {PROJECT_ROOT}")
print(f"Scripts dir:   {SCRIPTS_DIR}")


## 3. Experimental Protocol

The report discussion focuses on one-seed Waypoints diagnostics. The baseline mode comparison and delayed-learning runs use the original SAC report settings, while the reward-shaping runs reuse the later mode-0 hyperparameters (`learning_starts = 300k`, `buffer_size = 500k`) so that changes come from the reward rather than from a different training regime.


## 4. Shared Settings


In [ ]:
# ---- Output and execution flags ---------------------------------------------
RESULTS_ROOT = PROJECT_ROOT / "results"
FORCE_RETRAIN = False
PLOT_ONLY = False

# ---- Notebook sections to run -----------------------------------------------
RUN_MODE_COMPARISON = True
RUN_DELAYED_LEARNING = False
RUN_REWARD_VARIANTS = False


## 5. Configurations


In [ ]:
from dataclasses import replace

from training_cell_waypoints_sac import WaypointsTrainingConfig, list_reward_variants, output_paths, train_all

BASE_CFG = WaypointsTrainingConfig(
    algo_name="SAC",
    env_name="waypoints",
    flight_mode=0,
    timesteps=1_000_000,
    learning_rate=5e-5,
    buffer_size=1_000_000,
    batch_size=512,
    gamma=0.995,
    ent_coef="auto",
    learning_starts=50_000,
    gradient_steps=1,
    seeds=[0],
    eval_freq=10_000,
    n_eval_episodes=30,
    final_eval_episodes=50,
    project_root=PROJECT_ROOT,
    results_dir=RESULTS_ROOT / "waypoints_sac",
    save_name="SAC_waypoints_mode0_1_000_000steps",
)

MODE6_CFG = replace(
    BASE_CFG,
    flight_mode=6,
    save_name="SAC_waypoints_mode6_1_000_000steps_seed0_curve",
)

DELAYED_300K_CFG = replace(
    BASE_CFG,
    learning_starts=300_000,
    buffer_size=500_000,
    save_name="SAC_waypoints_mode0_1_000_000steps_ls300k_buf500k",
)

DELAYED_600K_CFG = replace(
    BASE_CFG,
    learning_starts=600_000,
    buffer_size=1_000_000,
    save_name="SAC_waypoints_mode0_1_000_000steps_ls600k_buf1000k",
)

REWARD_BASE = replace(
    BASE_CFG,
    learning_starts=300_000,
    buffer_size=500_000,
    n_eval_episodes=20,
    final_eval_episodes=20,
)

REWARD_DISTANCE_CFG = replace(
    REWARD_BASE,
    reward_variant="distance_dense",
    results_dir=RESULTS_ROOT / "waypoints_reward_shaped",
    save_name="SAC_waypoints_reward_shaped_mode0_1_000_000steps",
)

REWARD_RELPROGRESS_CFG = replace(
    REWARD_BASE,
    reward_variant="relative_progress",
    results_dir=RESULTS_ROOT / "waypoints_reward_shaped_relprogress",
    save_name="SAC_waypoints_reward_shaped_relprogress_mode0_1_000_000steps",
)

REWARD_SIGNED_CFG = replace(
    REWARD_BASE,
    reward_variant="signed_progress",
    results_dir=RESULTS_ROOT / "waypoints_reward_shaped_signedprogress",
    save_name="SAC_waypoints_reward_shaped_signedprogress_mode0_1_000_000steps",
)

REWARD_SOFTSIGNED_CFG = replace(
    REWARD_BASE,
    reward_variant="soft_signed",
    results_dir=RESULTS_ROOT / "waypoints_reward_shaped_softsigned",
    save_name="SAC_waypoints_reward_shaped_softsigned_mode0_1_000_000steps",
)

MODE_EXPERIMENTS = {
    "mode0": BASE_CFG,
    "mode6": MODE6_CFG,
}

DELAYED_EXPERIMENTS = {
    "ls300k_buf500k": DELAYED_300K_CFG,
    "ls600k_buf1000k": DELAYED_600K_CFG,
}

REWARD_EXPERIMENTS = {
    "distance_dense": REWARD_DISTANCE_CFG,
    "relative_progress": REWARD_RELPROGRESS_CFG,
    "signed_progress": REWARD_SIGNED_CFG,
    "soft_signed": REWARD_SOFTSIGNED_CFG,
}

print("Reward variants:", list_reward_variants())
for family_name, family in [("modes", MODE_EXPERIMENTS), ("delayed", DELAYED_EXPERIMENTS), ("rewards", REWARD_EXPERIMENTS)]:
    print(f"\n[{family_name}]")
    for name, cfg in family.items():
        print(name, cfg.resolved_save_name(), cfg.reward_variant, cfg.learning_starts, cfg.buffer_size)


## 6. Run Baseline Mode Comparison


In [ ]:
if RUN_MODE_COMPARISON:
    for name, cfg in MODE_EXPERIMENTS.items():
        print(f"\n===== Running {name} =====")
        train_all(cfg, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)


## 7. Run Delayed-Learning Diagnostics


In [ ]:
if RUN_DELAYED_LEARNING:
    for name, cfg in DELAYED_EXPERIMENTS.items():
        print(f"\n===== Running {name} =====")
        train_all(cfg, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)


## 8. Run Reward Variants


In [ ]:
if RUN_REWARD_VARIANTS:
    for name, cfg in REWARD_EXPERIMENTS.items():
        print(f"\n===== Running {name} =====")
        train_all(cfg, plot_only=PLOT_ONLY, force_retrain=FORCE_RETRAIN)


## 9. Inspect Outputs


In [ ]:
import pandas as pd
from IPython.display import Image, display

def display_artifacts(cfg):
    paths = output_paths(cfg)
    for name, path in paths.items():
        print(f"{name}: {path}  exists={path.exists()}")

    if paths["learning_curve"].exists():
        display(Image(filename=str(paths["learning_curve"])))
    if paths["final_boxplot"].exists():
        display(Image(filename=str(paths["final_boxplot"])))
    if paths["final_stats"].exists():
        display(pd.read_csv(paths["final_stats"]))
    if paths["best_stats"].exists():
        display(pd.read_csv(paths["best_stats"]))
    if paths["checkpoint_comparison"].exists():
        display(pd.read_csv(paths["checkpoint_comparison"]))


In [ ]:
# Examples:
display_artifacts(BASE_CFG)
# display_artifacts(MODE6_CFG)
# display_artifacts(DELAYED_300K_CFG)
# display_artifacts(DELAYED_600K_CFG)
# display_artifacts(REWARD_DISTANCE_CFG)
# display_artifacts(REWARD_RELPROGRESS_CFG)
# display_artifacts(REWARD_SIGNED_CFG)
# display_artifacts(REWARD_SOFTSIGNED_CFG)
